In [1]:
# ============================================================
# BOCCONI STUDENT SURVEY — INITIAL DATA STANDARDISATION
# Google Colab script
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import unicodedata
from google.colab import files


# ============================================================
# 2. UPLOAD AND READ THE EXCEL FILE
# ============================================================

print("Upload the original Survey_anagrafiche.xlsx file:")
uploaded = files.upload()

# Automatically retrieve the uploaded filename
input_filename = next(iter(uploaded))

# Read the survey sheet
sheet_name = "risposte"
df = pd.read_excel(input_filename, sheet_name=sheet_name)

print(f"\nFile loaded: {input_filename}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nColumn names:")
print(df.columns.tolist())


# ============================================================
# 3. FUNCTION USED TO MATCH INCONSISTENT CATEGORIES
# ============================================================

def normalize_key(value):
    """
    Creates a normalized version of a category for matching purposes:
    - removes leading/trailing spaces;
    - converts to lowercase;
    - removes accents;
    - removes repeated internal spaces.

    It does not directly modify the original value.
    """
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        character
        for character in value
        if not unicodedata.combining(character)
    )
    value = " ".join(value.split())

    return value


# ============================================================
# 4. CATEGORY-SPECIFIC STANDARDISATION RULES
# ============================================================

# Keys must be written in the normalized form produced by normalize_key().
# Categories not included here are preserved, except for excess whitespace.

category_mappings = {
    "Programma": {
        "data science": "Data Science",
        "datascience": "Data Science",
    },

    "Situazione_abitativa": {
        "affitto a milano": "Affitto a Milano",
        "affitto milano": "Affitto a Milano",
        "private rental milan": "Affitto a Milano",
    },

    "Lavora_part_time": {
        "si": "Sì",
        "yes": "Sì",
        "no": "No",
    },

    "Borsa_o_aiuto_economico": {
        "si": "Sì",
        "yes": "Sì",
        "no": "No",
    },

    "Conoscenza_servizi_supporto": {
        "si": "Sì",
        "yes": "Sì",
        "no": "No",
        "parzialmente": "Parzialmente",
        "partially": "Parzialmente",
    },

    "Uso_servizi_supporto": {
        "si": "Sì",
        "yes": "Sì",
        "no": "No",
    },
}


# ============================================================
# 5. IDENTIFY CATEGORICAL COLUMNS
# ============================================================

# Automatically identify text-based columns.
categorical_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# ID should normally remain an identifier rather than be standardized.
categorical_columns = [
    column for column in categorical_columns
    if column.lower() != "id"
]

print("\nCategorical columns identified:")
print(categorical_columns)


# ============================================================
# 6. CREATE STANDARDIZED COLUMNS
# ============================================================

for column in categorical_columns:

    standardized_column = f"{column}_std"

    # Remove unnecessary whitespace while preserving missing values
    cleaned_original = df[column].apply(
        lambda value: (
            " ".join(str(value).strip().split())
            if pd.notna(value)
            else pd.NA
        )
    )

    normalized_values = df[column].apply(normalize_key)

    if column in category_mappings:

        # Use explicit mappings where available
        standardized_values = normalized_values.map(
            category_mappings[column]
        )

        # Preserve legitimate categories not included in the mapping
        standardized_values = standardized_values.where(
            standardized_values.notna(),
            cleaned_original
        )

    else:
        # For columns with no known inconsistency, only clean whitespace
        standardized_values = cleaned_original

    # Insert the standardized column directly after the original one
    original_position = df.columns.get_loc(column)

    df.insert(
        original_position + 1,
        standardized_column,
        standardized_values
    )


# ============================================================
# 7. CONVERT US GPA VALUES INTO THE BOCCONI SCALE
# ============================================================

grade_column = "Media_voti"

if grade_column in df.columns:

    # Ensure the original column is interpreted numerically
    original_grades = pd.to_numeric(
        df[grade_column],
        errors="coerce"
    )

    # Start by preserving all existing valid Italian-scale grades
    converted_grades = original_grades.copy()

    # Agreed approximate conversion
    gpa_conversion = {
        3.5: 28.0,
        3.8: 29.0,
        4.0: 30.0,
    }

    # Use isclose to avoid floating-point matching problems
    conversion_flag = pd.Series(
        False,
        index=df.index
    )

    for american_gpa, bocconi_grade in gpa_conversion.items():

        rows_to_convert = np.isclose(
            original_grades,
            american_gpa,
            equal_nan=False
        )

        converted_grades.loc[rows_to_convert] = bocconi_grade
        conversion_flag.loc[rows_to_convert] = True

    standardized_grade_column = "Media_voti_std"
    grade_flag_column = "Media_voti_conversione_GPA_US"

    original_position = df.columns.get_loc(grade_column)

    # Add converted grade immediately after the original
    df.insert(
        original_position + 1,
        standardized_grade_column,
        converted_grades
    )

    # Add a transparent conversion flag
    df.insert(
        original_position + 2,
        grade_flag_column,
        conversion_flag.map({
            True: "Sì",
            False: "No"
        })
    )

    print("\nUS GPA conversions performed:")
    print(
        df.loc[
            conversion_flag,
            [
                grade_column,
                standardized_grade_column,
                grade_flag_column
            ]
        ]
    )

else:
    print(
        f"\nWarning: column '{grade_column}' was not found. "
        "No GPA conversion was performed."
    )


# ============================================================
# 8. PRODUCE A CATEGORY-CLEANING REPORT
# ============================================================

report_rows = []

for column in categorical_columns:

    standardized_column = f"{column}_std"

    changed = (
        df[column].fillna("<MISSING>").astype(str)
        !=
        df[standardized_column].fillna("<MISSING>").astype(str)
    )

    report_rows.append({
        "Original_column": column,
        "Standardized_column": standardized_column,
        "Changed_rows": int(changed.sum()),
        "Original_unique_values": int(df[column].nunique(dropna=True)),
        "Standardized_unique_values": int(
            df[standardized_column].nunique(dropna=True)
        )
    })

cleaning_report = pd.DataFrame(report_rows)

print("\nStandardisation report:")
display(cleaning_report)


# ============================================================
# 9. EXPORT THE CLEANED DATASET AND REPORT
# ============================================================

output_filename = "Survey_anagrafiche_standardizzato.xlsx"

with pd.ExcelWriter(
    output_filename,
    engine="openpyxl"
) as writer:

    # Cleaned quantitative dataset
    df.to_excel(
        writer,
        sheet_name="risposte_standardizzate",
        index=False
    )

    # Audit report
    cleaning_report.to_excel(
        writer,
        sheet_name="report_standardizzazione",
        index=False
    )

print(f"\nCleaned file created: {output_filename}")

# Automatically download the result
files.download(output_filename)

Upload the original Survey_anagrafiche.xlsx file:


Saving Survey_anagrafiche.xlsx to Survey_anagrafiche (1).xlsx

File loaded: Survey_anagrafiche (1).xlsx
Rows: 1,075
Columns: 24

Column names:
['ID', 'Data_risposta', 'Programma', 'Livello', 'Anno_corso', 'Origine', 'Area_geografica', 'Lingua_risposta', 'Residenza', 'Distanza_km', 'Tempo_pendolarismo_min', 'Situazione_abitativa', 'Lavora_part_time', 'Ore_lavoro_sett', 'Borsa_o_aiuto_economico', 'Pressione_economica_1_10', 'Media_voti', 'Crediti_conseguiti', 'Conoscenza_servizi_supporto', 'Uso_servizi_supporto', 'Soddisfazione_1_10', 'Benessere_1_10', 'Appartenenza_1_10', 'Carico_gestibile_1_10']

Categorical columns identified:
['Data_risposta', 'Programma', 'Livello', 'Origine', 'Area_geografica', 'Lingua_risposta', 'Residenza', 'Situazione_abitativa', 'Lavora_part_time', 'Borsa_o_aiuto_economico', 'Conoscenza_servizi_supporto', 'Uso_servizi_supporto']

US GPA conversions performed:
     Media_voti  Media_voti_std Media_voti_conversione_GPA_US
366         3.8            29.0          

,Original_column,Standardized_column,Changed_rows,Original_unique_values,Standardized_unique_values
0,Data_risposta,Data_risposta_std,0,28,28
1,Programma,Programma_std,9,10,7
2,Livello,Livello_std,0,2,2
3,Origine,Origine_std,0,2,2
4,Area_geografica,Area_geografica_std,0,5,5
5,Lingua_risposta,Lingua_risposta_std,0,3,3
6,Residenza,Residenza_std,0,5,5
7,Situazione_abitativa,Situazione_abitativa_std,8,6,4
8,Lavora_part_time,Lavora_part_time_std,162,5,2
9,Borsa_o_aiuto_economico,Borsa_o_aiuto_economico_std,162,5,2



Cleaned file created: Survey_anagrafiche_standardizzato.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# ============================================================
# 1. IMPORT PACKAGES
# ============================================================

import pandas as pd
from google.colab import files
from IPython.display import display


# ============================================================
# 2. UPLOAD QUANTITATIVE EXCEL
# ============================================================

print("Upload the quantitative Excel file")
uploaded_quant = files.upload()
quant_filename = next(iter(uploaded_quant))

print(f"Loaded: {quant_filename}")


# ============================================================
# 3. UPLOAD OPEN-RESPONSES EXCEL
# ============================================================

print("\nUpload the open-responses Excel file")
uploaded_open = files.upload()
open_filename = next(iter(uploaded_open))

print(f"Loaded: {open_filename}")


# ============================================================
# 4. READ BOTH EXCEL FILES
# ============================================================

# Quantitative dataset
quant_df = pd.read_excel(
    quant_filename,
    sheet_name="risposte"
)

# Open-response dataset
# If the file has only one sheet, this works directly.
open_df = pd.read_excel(open_filename)

print("\nQuantitative shape:", quant_df.shape)
print("Open-response shape:", open_df.shape)

print("\nQuantitative columns:")
print(quant_df.columns.tolist())

print("\nOpen-response columns:")
print(open_df.columns.tolist())


# ============================================================
# 5. KEEP ONLY RELEVANT QUANTITATIVE COLUMNS
# ============================================================

quant_columns_to_keep = [
    "ID",

    # Profile
    "Programma",
    "Livello",
    "Anno_corso",
    "Origine",
    "Lingua_risposta",

    # Economic pressure
    "Lavora_part_time",
    "Ore_lavoro_sett",
    "Borsa_o_aiuto_economico",
    "Pressione_economica_1_10",

    # Commuting and housing
    "Residenza",
    "Distanza_km",
    "Tempo_pendolarismo_min",
    "Situazione_abitativa",

    # Services
    "Conoscenza_servizi_supporto",
    "Uso_servizi_supporto",

    # Academic variables
    "Media_voti",
    "Crediti_conseguiti",

    # Main outcomes
    "Soddisfazione_1_10",
    "Benessere_1_10",
    "Appartenenza_1_10",
    "Carico_gestibile_1_10"
]

missing_quant_columns = [
    col for col in quant_columns_to_keep
    if col not in quant_df.columns
]

if missing_quant_columns:
    raise ValueError(
        f"Missing quantitative columns: {missing_quant_columns}"
    )

quant_selected = quant_df[quant_columns_to_keep].copy()


# ============================================================
# 6. CHECK OPEN-RESPONSE COLUMNS
# ============================================================

expected_open_columns = [
    "ID",
    "Difficolta_principale",
    "Cosa_cambieresti",
    "Cosa_funziona"
]

missing_open_columns = [
    col for col in expected_open_columns
    if col not in open_df.columns
]

if missing_open_columns:
    raise ValueError(
        f"Missing open-response columns: {missing_open_columns}"
    )

open_selected = open_df[expected_open_columns].copy()


# ============================================================
# 7. CLEAN IDS
# ============================================================

def clean_id(series):
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

quant_selected["ID"] = clean_id(quant_selected["ID"])
open_selected["ID"] = clean_id(open_selected["ID"])


# ============================================================
# 8. CHECK DUPLICATES
# ============================================================

print("\nDuplicate IDs in quantitative data:",
      quant_selected["ID"].duplicated().sum())

print("Duplicate IDs in open responses:",
      open_selected["ID"].duplicated().sum())


# ============================================================
# 9. MERGE BY ID
# ============================================================

merged_df = pd.merge(
    quant_selected,
    open_selected,
    on="ID",
    how="inner",
    validate="one_to_one"
)

print("\nMerged dataset shape:", merged_df.shape)
print("Matched respondents:", merged_df["ID"].nunique())


# ============================================================
# 10. CHECK UNMATCHED IDS
# ============================================================

quant_ids = set(quant_selected["ID"])
open_ids = set(open_selected["ID"])

only_quant = quant_ids - open_ids
only_open = open_ids - quant_ids

print("\nIDs only in quantitative dataset:", len(only_quant))
print("IDs only in open-response dataset:", len(only_open))


# ============================================================
# 11. CREATE USEFUL GROUP VARIABLES
# ============================================================

merged_df["Pressione_economica_gruppo"] = pd.cut(
    merged_df["Pressione_economica_1_10"],
    bins=[0, 4, 7, 10],
    labels=["Bassa_1_4", "Media_5_7", "Alta_8_10"],
    include_lowest=True
)

merged_df["Benessere_gruppo"] = pd.cut(
    merged_df["Benessere_1_10"],
    bins=[0, 4, 7, 10],
    labels=["Basso_1_4", "Medio_5_7", "Alto_8_10"],
    include_lowest=True
)

merged_df["Appartenenza_gruppo"] = pd.cut(
    merged_df["Appartenenza_1_10"],
    bins=[0, 4, 7, 10],
    labels=["Bassa_1_4", "Media_5_7", "Alta_8_10"],
    include_lowest=True
)

merged_df["Soddisfazione_gruppo"] = pd.cut(
    merged_df["Soddisfazione_1_10"],
    bins=[0, 4, 7, 10],
    labels=["Bassa_1_4", "Media_5_7", "Alta_8_10"],
    include_lowest=True
)

merged_df["Carico_gruppo"] = pd.cut(
    merged_df["Carico_gestibile_1_10"],
    bins=[0, 4, 7, 10],
    labels=["Poco_gestibile_1_4", "Medio_5_7", "Gestibile_8_10"],
    include_lowest=True
)

merged_df["Alta_soddisfazione_basso_benessere"] = (
    (merged_df["Soddisfazione_1_10"] >= 8) &
    (merged_df["Benessere_1_10"] <= 4)
)

merged_df["Alta_soddisfazione_carico_basso"] = (
    (merged_df["Soddisfazione_1_10"] >= 8) &
    (merged_df["Carico_gestibile_1_10"] <= 4)
)


# ============================================================
# 12. PREVIEW
# ============================================================

display(merged_df.head())

print("\nGroup counts:")
print(
    merged_df["Pressione_economica_gruppo"]
    .value_counts(dropna=False)
)


# ============================================================
# 13. EXPORT
# ============================================================

output_filename = "survey_studenti_merged_qual_quant.xlsx"

merged_df.to_excel(
    output_filename,
    index=False
)

print(f"\nCreated: {output_filename}")

files.download(output_filename)

Upload the quantitative Excel file


Saving Survey_anagrafiche.xlsx to Survey_anagrafiche (4).xlsx
Loaded: Survey_anagrafiche (4).xlsx

Upload the open-responses Excel file


Saving survey_studenti_risposte_aperte_UTF8.xlsx to survey_studenti_risposte_aperte_UTF8.xlsx
Loaded: survey_studenti_risposte_aperte_UTF8.xlsx

Quantitative shape: (1075, 24)
Open-response shape: (1200, 4)

Quantitative columns:
['ID', 'Data_risposta', 'Programma', 'Livello', 'Anno_corso', 'Origine', 'Area_geografica', 'Lingua_risposta', 'Residenza', 'Distanza_km', 'Tempo_pendolarismo_min', 'Situazione_abitativa', 'Lavora_part_time', 'Ore_lavoro_sett', 'Borsa_o_aiuto_economico', 'Pressione_economica_1_10', 'Media_voti', 'Crediti_conseguiti', 'Conoscenza_servizi_supporto', 'Uso_servizi_supporto', 'Soddisfazione_1_10', 'Benessere_1_10', 'Appartenenza_1_10', 'Carico_gestibile_1_10']

Open-response columns:
['ID', 'Difficolta_principale', 'Cosa_cambieresti', 'Cosa_funziona']

Duplicate IDs in quantitative data: 0
Duplicate IDs in open responses: 0

Merged dataset shape: (1075, 25)
Matched respondents: 1075

IDs only in quantitative dataset: 0
IDs only in open-response dataset: 125


,ID,Programma,Livello,Anno_corso,Origine,Lingua_risposta,Lavora_part_time,Ore_lavoro_sett,Borsa_o_aiuto_economico,Pressione_economica_1_10,...,Difficolta_principale,Cosa_cambieresti,Cosa_funziona,Pressione_economica_gruppo,Benessere_gruppo,Appartenenza_gruppo,Soddisfazione_gruppo,Carico_gruppo,Alta_soddisfazione_basso_benessere,Alta_soddisfazione_carico_basso
0,STU_0003,Data Science,Bachelor,2,Internazionale,Mixed,No,0.0,No,7.0,...,NaN,boh,boh,Media_5_7,Basso_1_4,Media_5_7,Alta_8_10,Poco_gestibile_1_4,True,True
1,STU_0005,Law,Bachelor,2,Italia,Italiano,No,0.0,No,4.0,...,boh,NaN,non saprei,Bassa_1_4,Basso_1_4,Media_5_7,Media_5_7,Poco_gestibile_1_4,False,False
2,STU_0006,Law,Bachelor,2,Italia,Italiano,no,0.0,No,3.0,...,Forse lo stress dei tre esami in dieci giorni.,Attività e gruppi in fasce orarie realistiche ...,"Per me difficile dirlo adesso, forse lo capirò...",Bassa_1_4,Alto_8_10,Media_5_7,Media_5_7,Medio_5_7,False,False
3,STU_0007,Marketing,Bachelor,3,Italia,Italiano,No,0.0,No,6.0,...,Lo stress dei tre esami in dieci giorni. La pr...,Esami che premino il ragionamento e non le sli...,"Boh, difficile dirlo adesso, forse lo capirò d...",Media_5_7,Basso_1_4,Media_5_7,Media_5_7,Medio_5_7,False,False
4,STU_0010,International Politics,Bachelor,1,Italia,Italiano,No,0.0,No,3.0,...,"Boh, i gruppi di lavoro sbilanciati, finisco a...",Esami che premino il ragionamento e non le sli...,NaN,Bassa_1_4,Medio_5_7,Alta_8_10,Alta_8_10,Poco_gestibile_1_4,False,True



Group counts:
Pressione_economica_gruppo
Media_5_7    497
Bassa_1_4    423
Alta_8_10    147
NaN            8
Name: count, dtype: int64

Created: survey_studenti_merged_qual_quant.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
themes = {
    "Carico_esami": [
        "esami", "carico", "studio", "deadline", "consegne",
        "sessione", "stress", "pressione", "workload", "assignments"
    ],

    "Organizzazione_orari": [
        "orari", "orario", "buchi", "lezioni sparse",
        "sovrapposizioni", "timetable", "schedule"
    ],

    "Pressione_performance": [
        "voti", "media", "competizione", "cv", "stage",
        "prestazione", "performance", "non basta mai"
    ],

    "Pendolarismo_trasporti": [
        "pendolare", "treno", "metro", "autobus", "mezzi",
        "trasporto", "viaggio", "distanza", "arrivare da fuori"
    ],

    "Costi_pressione_economica": [
        "costo", "costi", "affitto", "retta", "soldi",
        "caro", "economico", "expensive", "rent", "tuition"
    ],

    "Lavoro_studio": [
        "lavoro", "lavorare", "part-time", "job",
        "working", "turni"
    ],

    "Socialita_appartenenza": [
        "amici", "amicizie", "conoscere persone", "socialità",
        "community", "comunità", "integrazione", "isolamento",
        "friends", "belonging"
    ],

    "Internazionali_lingua": [
        "lingua", "italiano", "inglese", "language",
        "international", "internazionale", "cultura"
    ],

    "Salute_stress": [
        "ansia", "stress", "stanco", "stanchezza",
        "salute mentale", "benessere", "burnout",
        "anxiety", "mental health", "exhausted"
    ],

    "Didattica_valutazione": [
        "professori", "docenti", "insegnamento", "slide",
        "memoria", "gruppi di lavoro", "esami che premino",
        "teaching", "professor"
    ]
}

In [7]:
import pandas as pd
import re
import unicodedata

file_path = "/content/survey_studenti_merged_qual_quant.xlsx"

df = pd.read_excel(file_path)

themes = {
    "Carico_esami": [
        "esami", "carico", "studio", "deadline", "consegne",
        "sessione", "stress", "workload", "assignments"
    ],
    "Organizzazione_orari": [
        "orari", "orario", "buchi", "lezioni sparse",
        "sovrapposizioni", "timetable", "schedule"
    ],
    "Pressione_performance": [
        "voti", "media", "competizione", "cv", "stage",
        "prestazione", "performance", "non basta mai"
    ],
    "Pendolarismo_trasporti": [
        "pendolare", "treno", "metro", "autobus", "mezzi",
        "trasporto", "viaggio", "distanza", "arrivare da fuori"
    ],
    "Costi_pressione_economica": [
        "costo", "costi", "affitto", "retta", "soldi",
        "caro", "economico", "expensive", "rent", "tuition"
    ],
    "Lavoro_studio": [
        "lavoro", "lavorare", "part-time", "job",
        "working", "turni"
    ],
    "Socialita_appartenenza": [
        "amici", "amicizie", "conoscere persone", "socialita",
        "community", "comunita", "integrazione", "isolamento",
        "friends", "belonging"
    ],
    "Internazionali_lingua": [
        "lingua", "italiano", "inglese", "language",
        "international", "internazionale", "cultura"
    ],
    "Salute_stress": [
        "ansia", "stress", "stanco", "stanchezza",
        "salute mentale", "benessere", "burnout",
        "anxiety", "mental health", "exhausted"
    ],
    "Didattica_valutazione": [
        "professori", "docenti", "insegnamento", "slide",
        "memoria", "gruppi di lavoro", "esami che premino",
        "teaching", "professor"
    ]
}

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()

    text = unicodedata.normalize("NFD", text)
    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Mn"
    )

    return text

df["Difficolta_normalizzata"] = (
    df["Difficolta_principale"]
    .apply(normalize_text)
)

for theme, keywords in themes.items():

    pattern = "|".join(
        re.escape(normalize_text(keyword))
        for keyword in keywords
    )

    df[f"tema_{theme}"] = (
        df["Difficolta_normalizzata"]
        .str.contains(pattern, regex=True, na=False)
    )

theme_columns = [
    f"tema_{theme}"
    for theme in themes
]

df["Numero_temi_rilevati"] = df[theme_columns].sum(axis=1)

print("Theme frequencies:")
print(
    df[theme_columns]
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(1)
)

Theme frequencies:
tema_Carico_esami                 36.6
tema_Socialita_appartenenza        9.4
tema_Lavoro_studio                 9.1
tema_Salute_stress                 8.8
tema_Didattica_valutazione         6.0
tema_Pressione_performance         5.2
tema_Internazionali_lingua         2.0
tema_Pendolarismo_trasporti        2.0
tema_Costi_pressione_economica     1.9
tema_Organizzazione_orari          0.0
dtype: float64


In [8]:
financial_comparison = (
    df[
        df["Pressione_economica_gruppo"]
        .isin(["Bassa_1_4", "Alta_8_10"])
    ]
    .groupby("Pressione_economica_gruppo", observed=True)[theme_columns]
    .mean()
    .mul(100)
    .round(1)
    .T
)

financial_comparison["Differenza_alta_meno_bassa"] = (
    financial_comparison["Alta_8_10"]
    - financial_comparison["Bassa_1_4"]
)

financial_comparison = financial_comparison.sort_values(
    "Differenza_alta_meno_bassa",
    ascending=False
)

display(financial_comparison)

Pressione_economica_gruppo,Alta_8_10,Bassa_1_4,Differenza_alta_meno_bassa
tema_Lavoro_studio,16.3,7.3,9.0
tema_Salute_stress,11.6,9.0,2.6
tema_Costi_pressione_economica,2.7,1.2,1.5
tema_Internazionali_lingua,1.4,0.7,0.7
tema_Pendolarismo_trasporti,2.0,1.9,0.1
tema_Organizzazione_orari,0.0,0.0,0.0
tema_Carico_esami,39.5,39.7,-0.2
tema_Didattica_valutazione,4.8,6.4,-1.6
tema_Pressione_performance,2.7,4.7,-2.0
tema_Socialita_appartenenza,6.1,8.3,-2.2


In [9]:
belonging_comparison = (
    df[
        df["Appartenenza_gruppo"]
        .isin(["Bassa_1_4", "Alta_8_10"])
    ]
    .groupby("Appartenenza_gruppo", observed=True)[theme_columns]
    .mean()
    .mul(100)
    .round(1)
    .T
)

belonging_comparison["Differenza_bassa_meno_alta"] = (
    belonging_comparison["Bassa_1_4"]
    - belonging_comparison["Alta_8_10"]
)

belonging_comparison = belonging_comparison.sort_values(
    "Differenza_bassa_meno_alta",
    ascending=False
)

display(belonging_comparison)

Appartenenza_gruppo,Alta_8_10,Bassa_1_4,Differenza_bassa_meno_alta
tema_Socialita_appartenenza,5.2,14.1,8.9
tema_Pendolarismo_trasporti,0.0,3.8,3.8
tema_Costi_pressione_economica,0.8,3.4,2.6
tema_Internazionali_lingua,1.2,3.4,2.2
tema_Organizzazione_orari,0.0,0.0,0.0
tema_Pressione_performance,6.8,2.1,-4.7
tema_Salute_stress,11.6,6.4,-5.2
tema_Lavoro_studio,13.1,5.6,-7.5
tema_Didattica_valutazione,10.4,1.7,-8.7
tema_Carico_esami,45.0,28.6,-16.4


In [10]:
contradiction_comparison = (
    df.groupby(
        "Alta_soddisfazione_basso_benessere"
    )[theme_columns]
    .mean()
    .mul(100)
    .round(1)
    .T
)

contradiction_comparison.columns = [
    "Altri_studenti",
    "Alta_soddisfazione_basso_benessere"
]

contradiction_comparison["Differenza"] = (
    contradiction_comparison[
        "Alta_soddisfazione_basso_benessere"
    ]
    - contradiction_comparison["Altri_studenti"]
)

contradiction_comparison = contradiction_comparison.sort_values(
    "Differenza",
    ascending=False
)

display(contradiction_comparison)

,Altri_studenti,Alta_soddisfazione_basso_benessere,Differenza
tema_Costi_pressione_economica,1.7,3.0,1.3
tema_Pressione_performance,5.1,6.1,1.0
tema_Pendolarismo_trasporti,2.0,2.0,0.0
tema_Organizzazione_orari,0.0,0.0,0.0
tema_Internazionali_lingua,2.0,2.0,0.0
tema_Socialita_appartenenza,9.7,6.1,-3.6
tema_Didattica_valutazione,6.4,2.0,-4.4
tema_Carico_esami,37.0,32.3,-4.7
tema_Lavoro_studio,9.6,4.0,-5.6
tema_Salute_stress,9.4,3.0,-6.4


In [11]:
selected_comments = df.loc[
    (
        df["Pressione_economica_gruppo"] == "Alta_8_10"
    )
    &
    (
        df["tema_Costi_pressione_economica"]
    ),
    [
        "ID",
        "Pressione_economica_1_10",
        "Benessere_1_10",
        "Difficolta_principale",
        "Cosa_cambieresti",
        "Cosa_funziona"
    ]
]

display(selected_comments.head(30))

,ID,Pressione_economica_1_10,Benessere_1_10,Difficolta_principale,Cosa_cambieresti,Cosa_funziona
59,STU_0115,8.0,4,"Money. Rent plus living costs in Milan, and wo...",Wifi e prenotazioni che funzionino.,NaN
95,STU_0200,8.0,6,"Money. Rent plus living costs in Milan, and wo...",Flessibilità su appelli e registrazioni per ch...,"The opportunities are real, companies actually..."
617,STU_0052,8.0,4,"Money. Rent plus living costs in Milan, and wo...",Flessibilità su appelli e registrazioni per ch...,"The opportunities are real, companies actually..."
716,STU_0295,9.0,1,"Money. Rent plus living costs in Milan, and wo...",Wifi e prenotazioni che funzionino.,"Campus life is lively, the clubs and people ma..."


In [12]:
commuting_comments = df.loc[
    (
        df["Appartenenza_gruppo"] == "Bassa_1_4"
    )
    &
    (
        df["tema_Pendolarismo_trasporti"]
        |
        df["tema_Organizzazione_orari"]
    ),
    [
        "ID",
        "Distanza_km",
        "Tempo_pendolarismo_min",
        "Situazione_abitativa",
        "Appartenenza_1_10",
        "Difficolta_principale",
        "Cosa_cambieresti"
    ]
]

display(commuting_comments.head(30))

,ID,Distanza_km,Tempo_pendolarismo_min,Situazione_abitativa,Appartenenza_1_10,Difficolta_principale,Cosa_cambieresti
19,STU_0038,53.0,64.0,Famiglia,3.0,Direi le ore sul treno che non ti torna nessun...,"Forse orari più compatti, per un pendolare i b..."
177,STU_0358,46.0,71.0,Famiglia,4.0,"Trenord permettendo, ma soprattutto il fatto c...",Esami che premino il ragionamento e non le sli...
188,STU_0372,40.0,91.0,Famiglia,4.0,"Onestamente trenord permettendo, ma soprattutt...",Forse concentrare le lezioni su meno giorni.
304,STU_0616,66.0,76.0,Famiglia,4.0,"Sto tanto in treno e poco a Milano, e si sente...",Forse wifi e prenotazioni che funzionino.
509,STU_1020,24.0,56.0,Famiglia,4.0,"Due ore e mezza di viaggio al giorno, alla fin...",Wifi e prenotazioni che funzionino.
580,STU_1154,35.0,76.0,Famiglia,3.0,"Sto tanto in treno e poco a Milano, e si sente...","Lezioni registrate, così i giorni che salto il..."
728,STU_0329,38.0,57.0,Famiglia,4.0,"Sinceramente vivo fuori e faccio la pendolare,...","Gruppi formati con criterio, non a caso."
959,STU_0885,29.0,59.0,Famiglia,1.0,"Per me vivo fuori e faccio la pendolare, mi pe...",Forse concentrare le lezioni su meno giorni.
982,STU_0934,65.0,106.0,Famiglia,4.0,"Guarda, trenord permettendo, ma soprattutto il...","Lezioni registrate, così i giorni che salto il..."


In [13]:
output_path = "/content/survey_studenti_coded.xlsx"

with pd.ExcelWriter(output_path) as writer:
    df.to_excel(
        writer,
        sheet_name="Dati_codificati",
        index=False
    )

    financial_comparison.to_excel(
        writer,
        sheet_name="Pressione_economica"
    )

    belonging_comparison.to_excel(
        writer,
        sheet_name="Appartenenza"
    )

    contradiction_comparison.to_excel(
        writer,
        sheet_name="Soddisfazione_benessere"
    )

from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>